# 05 · LM likelihood features (for X3) · GPU

Two open-weight causal LMs that share a tokenizer (observer `Qwen2.5-0.5B`, performer `Qwen2.5-0.5B-Instruct`) score every text: token log-probabilities, observer entropy, observer→performer cross-entropy and the Binoculars ratio, plus their spread over 64-token segments (a partly rewritten text should mix segments of different likelihood). Token budget per language comes from `01_eda`.

Features are computed for a train subsample (`train_docs`), the validation split and the test set, saved to `artifacts/features/`, and classified in `06_experiments` (CPU). Run options are the same as in `04_encoder` (`uv run python -m hw3.lm` locally).

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/kerfmanner/ai-text-detection-hw3-ucu-ai.git"
CLOUD = "COLAB_RELEASE_TAG" in os.environ or "KAGGLE_KERNEL_RUN_TYPE" in os.environ
if CLOUD:
    if not Path("ai-text-detection-hw3-ucu-ai").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    os.chdir("ai-text-detection-hw3-ucu-ai")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==5.24.0"], check=True)
    sys.path.insert(0, str(Path("src").resolve()))
CLOUD

In [ ]:
SMOKE = False

import pandas as pd
import torch

from hw3 import config, lm
from hw3.data import download, load_train
from hw3.runs import export_outputs

config.set_smoke(SMOKE)
download()
cfg = config.load_config("lm_features")
torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", cfg

## Optional: crop ablation (run before the full feature run)

Texts longer than the token budget are cropped. `01_eda` shows German fusion texts keep most of their human source at the start and the least in the middle, so the LM may see more of the rewritten part in a middle crop. Both crops are computed on a smaller sample (`lm_crop_*.yaml`: 10k train, 6k val, 6k test) into separate artifact folders, then compared on CPU. If `middle` wins, set `crop: middle` in `lm_features.yaml` and re-run the config cell above before the full run below. About 45–60 min on a T4.

In [ ]:
base = Path(os.environ.get("HW3_ARTIFACTS", config.ROOT / "artifacts"))
for name in ["lm_crop_start", "lm_crop_middle"]:
    os.environ["HW3_ARTIFACTS"] = str(base / name)
    lm.run_features(config.load_config(name))
    print(export_outputs(name, ["features/*", "runs/*", "logs/*"]))
os.environ["HW3_ARTIFACTS"] = str(base)

In [ ]:
lm.run_features(cfg)

Median of selected features per language and class on the validation split.

In [ ]:
feats = pd.read_parquet(config.art("features", "lm_val.parquet")).merge(load_train()[["id", "lang", "label"]], on="id")
feats.groupby(["lang", "label"])[["lp_obs_mean", "ent_obs_mean", "binoculars", "seg_bino_std", "seg_lp_std"]].median().round(3)

In [ ]:
export_outputs("X3_features_outputs", ["features/lm_*", "runs/X3_features*", "logs/X3_features*"])